In [2]:
%matplotlib qt

In [3]:
import kwant
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from scipy.linalg import eig
import mplcursors
from matplotlib.colors import Normalize, TwoSlopeNorm


In [4]:
#参数
dela = 0.1875
t = 38*dela
af =1
a = 1
mu = 8*dela


hc=1.2#100
h =hc * np.sqrt(mu**2 + dela**2)
hz=0*dela
chaodaojiao = 0
#saimanjiao=np.pi/6
saimanjiao = np.pi*0


U = 1.9*dela
mz = 0.098614#0.098614 #【0.422270(mc=0,af=0，最小0)】,【0.098614(mc=1,af=1,最大1)】
mc=1
m =  h*mc
zhongjian_saimanjiao=np.pi*0
af6666=1

T_Tc=0.5
Delta = dela * np.tanh(1.74 * np.sqrt(1 / T_Tc - 1))

N=10
q=3
shumu=int(np.floor(5 / T_Tc))
KBT=2 * dela * T_Tc / 3.5

#矩阵信息
sx = np.array([[0, 1], [1, 0]], complex)
sy = np.array([[0, -1j], [1j, 0]], complex)
sz = np.array([[1, 0], [0, -1]], complex)
s0 = np.array([[1, 0], [0, 1]], complex)

#左边矩阵信息
HL_block=-(mu-2*t)*s0 + h*np.cos(0)*sx + h*np.sin(0)*sy +hz*sz
Delta_L=Delta * np.exp(+1j*chaodaojiao/2) * 1j * sy
H_L_onsite=np.block([
        [ HL_block,        Delta_L        ],
        [ Delta_L.conj().T, -HL_block.conj() ]
    ])
H_L_right_to_left_hop_block=-1*(t*s0+1j*af*sz/a)
H_L_right_to_left_hop=np.block([
        [ H_L_right_to_left_hop_block,        np.zeros((2,2))],
        [ np.zeros((2,2)), -H_L_right_to_left_hop_block.conj() ]
    ])

#中间矩阵信息
H_center_to_L=np.block([
        [ -1*t*s0,        np.zeros((2,2)) ],
        [ np.zeros((2,2)) , t*s0 ]
    ])


H_center_block=(U-mu+2*t)*s0 + mz*sz + m*np.cos(zhongjian_saimanjiao)*sx + m*np.sin(zhongjian_saimanjiao)*sy
H_center=np.block([
        [ H_center_block,        np.zeros((2,2)) ],
        [ np.zeros((2,2)) , -H_center_block.conj()  ]
    ])


H_center_right_to_left_hop_block=-1*(t*s0+1j*af*sz*af6666/a)
H_center_right_to_left_hop=np.block([
        [ H_center_right_to_left_hop_block,        np.zeros((2,2)) ],
        [ np.zeros((2,2)) , -H_center_right_to_left_hop_block.conj() ]
    ])


H_R_to_center=np.block([
        [ -t*s0,        np.zeros((2,2)) ],
        [ np.zeros((2,2)) , t*s0 ]
    ])

#右边矩阵信息
HR_block=-(mu-2*t)*s0 + h*np.cos(saimanjiao)*sx + h*np.sin(saimanjiao)*sy +hz*sz
Delta_R=Delta * np.exp(-1j*chaodaojiao/2) * 1j * sy
H_R_onsite=np.block([
        [ HR_block,        Delta_R        ],
        [ Delta_R.conj().T, -HR_block.conj() ]
    ])
H_R_right_to_left_hop_block=-1*(t*s0+1j*af/a*sz)
H_R_right_to_left_hop=np.block([
        [ H_R_right_to_left_hop_block,        np.zeros((2,2))],
        [ np.zeros((2,2)), -H_R_right_to_left_hop_block.conj() ]
    ])


#输入矩阵
H_q=H_center
T_12= H_center_right_to_left_hop.conj().T

H_l= H_L_onsite
T_l= H_L_right_to_left_hop

H_r= H_R_onsite
T_r=H_R_right_to_left_hop.conj().T

T_LD=H_center_to_L
T_RD= H_R_to_center



In [7]:
#右边
P_e = np.diag([1, 1, 0, 0])
P_h = np.diag([0, 0, 1, 1])
Sz = np.block([
    [sz, np.zeros((2,2))],
    [np.zeros((2,2)), sz]
])

ks = np.linspace(-np.pi, np.pi, 3001)

def Hk(k, H, T):
    return H + T * np.exp(1j*k) + T.conj().T * np.exp(-1j*k)


energies       = []
electron_weight = []
spin_expect    = []

for k in ks:
    H_k = Hk(k, H_r, T_r)
    evals, evecs = np.linalg.eigh(H_k)
    energies.append(evals)
    
    for n in range(len(evals)):
        psi = evecs[:, n]
        w_e = np.real(psi.conj().T @ P_e @ psi)
        s_z = np.real(psi.conj().T @ Sz @ psi)
        electron_weight.append(w_e)
        spin_expect.append(s_z)

energies       = np.array(energies)/Delta
electron_weight = np.array(electron_weight).reshape(len(ks), -1)
spin_expect    = np.array(spin_expect).reshape(len(ks), -1)

# ────────────────────────────────────────────────
# 第一张图：电子/空穴 character
# ────────────────────────────────────────────────
fig1, ax1 = plt.subplots(figsize=(7, 5))
ax1.set_facecolor('#f8f8f8')

norm_e = Normalize(vmin=0, vmax=1)

scatters_e = []
for n in range(energies.shape[1]):
    sc = ax1.scatter(
        ks, 
        energies[:, n],
        c=electron_weight[:, n],
        cmap='coolwarm',
        norm=norm_e,
        s=4,
        alpha=0.9,
        rasterized=True
    )
    sc.band_index = n          # 自定义属性，用于 hover 时识别是第几条带
    scatters_e.append(sc)

cbar1 = plt.colorbar(scatters_e[0], ax=ax1, label='Electron weight')
ax1.set_xlabel(r'$k$')
ax1.set_ylabel(f'E/$//Delta$')
ax1.set_title(
    f'$\\Delta = {dela:.4f}$ meV,  $\\mu = {mu/dela:.1f}\\Delta$,  $T/T_c = {T_Tc}$\n'
    f'$h_z = {hz/dela:.2f}\\Delta$, $h_c = {h/dela:.2f}\\Delta$,  $\\phi_1 = {saimanjiao / np.pi:.1f}\\pi$,  $\\alpha_1 = {af*10}$meVÅ\n'
    f'$m_z = {mz/dela:.2f}\\Delta$, $m_c = {m/dela:.2f}\\Delta$,  $\\phi_2 = {zhongjian_saimanjiao / np.pi:.1f}\\pi$,$U = {U/dela:.1f}\\Delta$,  $\\alpha_2 = {af*af6666*10}$meVÅ'
)
ax1.grid(True, alpha=0.4)

# hover 信息（只定义一次）
cursor_e = mplcursors.cursor(scatters_e, hover=True)
@cursor_e.connect("add")
def on_add_e(sel):
    sc = sel.artist
    band = sc.band_index
    i = sel.index
    sel.annotation.set_text(
        f"k     = {ks[i]:.4f}\n"
        f"E     = {energies[i, band]:.4f}\n"
        f"band  = {band}\n"
        f"electron = {electron_weight[i, band]:.3f}"
    )

# ────────────────────────────────────────────────
# 第二张图：自旋极化
# ────────────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(7, 5))
ax2.set_facecolor('#f8f8f8')

norm_s = TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)

scatters_s = []
for n in range(energies.shape[1]):
    sc = ax2.scatter(
        ks,
        energies[:, n],
        c=spin_expect[:, n],
        cmap='bwr',
        norm=norm_s,
        s=4,
        alpha=0.9,
        rasterized=True
    )
    sc.band_index = n
    scatters_s.append(sc)

cbar2 = plt.colorbar(scatters_s[0], ax=ax2, label=r'$\langle S_z \rangle$')
ax2.set_xlabel(r'$k$')
ax2.set_ylabel('Energy')
ax2.set_title(
    f'$\\Delta = {dela:.4f}$ meV,  $\\mu = {mu/dela:.1f}\\Delta$,  $T/T_c = {T_Tc}$\n'
    f'$h_z = {hz/dela:.2f}\\Delta$, $h_c = {h/dela:.2f}\\Delta$,  $\\phi_1 = {saimanjiao / np.pi:.1f}\\pi$,  $\\alpha_1 = {af*10}$meVÅ\n'
    f'$m_z = {mz/dela:.2f}\\Delta$, $m_c = {m/dela:.2f}\\Delta$,  $\\phi_2 = {zhongjian_saimanjiao / np.pi:.1f}\\pi$,$U = {U/dela:.1f}\\Delta$,  $\\alpha_2 = {af*af6666*10}$meVÅ'
)
ax2.grid(True, alpha=0.4)
ax2.set_ylim(-3, 3)    

cursor_s = mplcursors.cursor(scatters_s, hover=True)
@cursor_s.connect("add")
def on_add_s(sel):
    sc = sel.artist
    band = sc.band_index
    i = sel.index
    sel.annotation.set_text(
        f"k     = {ks[i]:.4f}\n"
        f"E     = {energies[i, band]:.4f}\n"
        f"band  = {band}\n"
        f"⟨S_z⟩ = {spin_expect[i, band]:.3f}"
    )

plt.show()

In [6]:
#中间
P_e = np.diag([1, 1, 0, 0])
P_h = np.diag([0, 0, 1, 1])
Sz = np.block([
    [sz, np.zeros((2,2))],
    [np.zeros((2,2)), sz]
])

ks = np.linspace(-np.pi, np.pi, 3001)

def Hk(k, H, T):
    return H + T * np.exp(1j*k) + T.conj().T * np.exp(-1j*k)


energies       = []
electron_weight = []
spin_expect    = []

for k in ks:
    H_k = Hk(k, H_q, T_12)
    evals, evecs = np.linalg.eigh(H_k)
    energies.append(evals)
    
    for n in range(len(evals)):
        psi = evecs[:, n]
        w_e = np.real(psi.conj().T @ P_e @ psi)
        s_z = np.real(psi.conj().T @ Sz @ psi)
        electron_weight.append(w_e)
        spin_expect.append(s_z)

energies       = np.array(energies)/Delta
electron_weight = np.array(electron_weight).reshape(len(ks), -1)
spin_expect    = np.array(spin_expect).reshape(len(ks), -1)

# ────────────────────────────────────────────────
# 第一张图：电子/空穴 character
# ────────────────────────────────────────────────
fig1, ax1 = plt.subplots(figsize=(7, 5))
ax1.set_facecolor('#f8f8f8')

norm_e = Normalize(vmin=0, vmax=1)

scatters_e = []
for n in range(energies.shape[1]):
    sc = ax1.scatter(
        ks, 
        energies[:, n],
        c=electron_weight[:, n],
        cmap='coolwarm',
        norm=norm_e,
        s=4,
        alpha=0.9,
        rasterized=True
    )
    sc.band_index = n          # 自定义属性，用于 hover 时识别是第几条带
    scatters_e.append(sc)

cbar1 = plt.colorbar(scatters_e[0], ax=ax1, label='Electron weight')
ax1.set_xlabel(r'$k$')
ax1.set_ylabel(f'E/$//Delta$')
ax1.set_title(
    f'$\\Delta = {dela:.4f}$ meV,  $\\mu = {mu/dela:.1f}\\Delta$,  $T/T_c = {T_Tc}$\n'
    f'$h_z = {hz/dela:.2f}\\Delta$, $h_c = {h/dela:.2f}\\Delta$,  $\\phi_1 = {saimanjiao / np.pi:.1f}\\pi$,  $\\alpha_1 = {af*10}$meVÅ\n'
    f'$m_z = {mz/dela:.2f}\\Delta$, $m_c = {m/dela:.2f}\\Delta$,  $\\phi_2 = {zhongjian_saimanjiao / np.pi:.1f}\\pi$,$U = {U/dela:.1f}\\Delta$,  $\\alpha_2 = {af*af6666*10}$meVÅ'
)

ax1.grid(True, alpha=0.4)

# hover 信息（只定义一次）
cursor_e = mplcursors.cursor(scatters_e, hover=True)
@cursor_e.connect("add")
def on_add_e(sel):
    sc = sel.artist
    band = sc.band_index
    i = sel.index
    sel.annotation.set_text(
        f"k     = {ks[i]:.4f}\n"
        f"E     = {energies[i, band]:.4f}\n"
        f"band  = {band}\n"
        f"electron = {electron_weight[i, band]:.3f}"
    )

# ────────────────────────────────────────────────
# 第二张图：自旋极化
# ────────────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(7, 5))
ax2.set_facecolor('#f8f8f8')

norm_s = TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)

scatters_s = []
for n in range(energies.shape[1]):
    sc = ax2.scatter(
        ks,
        energies[:, n],
        c=spin_expect[:, n],
        cmap='bwr',
        norm=norm_s,
        s=4,
        alpha=0.9,
        rasterized=True
    )
    sc.band_index = n
    scatters_s.append(sc)

cbar2 = plt.colorbar(scatters_s[0], ax=ax2, label=r'$\langle S_z \rangle$')
ax2.set_xlabel(r'$k$')
ax2.set_ylabel('Energy')
ax2.set_title(
    f'$\\Delta = {dela:.4f}$ meV,  $\\mu = {mu/dela:.1f}\\Delta$,  $T/T_c = {T_Tc}$\n'
    f'$h_z = {hz/dela:.2f}\\Delta$, $h_c = {h/dela:.2f}\\Delta$,  $\\phi_1 = {saimanjiao / np.pi:.1f}\\pi$,  $\\alpha_1 = {af*10}$meVÅ\n'
    f'$m_z = {mz/dela:.2f}\\Delta$, $m_c = {m/dela:.2f}\\Delta$,  $\\phi_2 = {zhongjian_saimanjiao / np.pi:.1f}\\pi$,$U = {U/dela:.1f}\\Delta$,  $\\alpha_2 = {af*af6666*10}$meVÅ'
)
ax2.grid(True, alpha=0.4)
ax2.set_ylim(-3, 3)   # 如果你确定这个范围合理

cursor_s = mplcursors.cursor(scatters_s, hover=True)
@cursor_s.connect("add")
def on_add_s(sel):
    sc = sel.artist
    band = sc.band_index
    i = sel.index
    sel.annotation.set_text(
        f"k     = {ks[i]:.4f}\n"
        f"E     = {energies[i, band]:.4f}\n"
        f"band  = {band}\n"
        f"⟨S_z⟩ = {spin_expect[i, band]:.3f}"
    )

plt.show()